# 参考mnist_conv-keras实现针对cifar10的alexNet卷积模型


#### 链接: https://pan.baidu.com/s/1LcCPcK9DgLS3W_DUPZS8kQ 提取码: 5vwz
### 解压放到 ~/.keras/datasets/

## tar zxvf cifar***.tar.zip

## 准备数据

In [ ]:
import os
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, optimizers, datasets
from tensorflow.keras.layers import Dense, Dropout, Flatten
from tensorflow.keras.layers import Conv2D, MaxPooling2D

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'  # or any {'0', '1', '2'}

def cifar10_dataset():
    (x, y), (x_test, y_test) = datasets.cifar10.load_data()
    ds = tf.data.Dataset.from_tensor_slices((x, y))
    ds = ds.map(prepare_mnist_features_and_labels)
    ds = ds.take(20000).shuffle(20000).batch(100)
    
    test_ds = tf.data.Dataset.from_tensor_slices((x_test, y_test))
    test_ds = test_ds.map(prepare_mnist_features_and_labels)
    test_ds = test_ds.take(20000).batch(20000)
    return ds, test_ds

def prepare_mnist_features_and_labels(x, y):
    x = tf.cast(x, tf.float32) / 255.0
    y = tf.cast(y, tf.int64)
    return x, y

## 建立模型

In [ ]:
class myConvModel(keras.Model):
    '''在这里实现alexNet模型'''
    def __init__(self):
        super(myConvModel, self).__init__()
        self.l1_conv = Conv2D(filters=32, 
                              kernel_size=(5, 5), 
                              activation='relu', padding='same')
        
        self.l2_conv = Conv2D(filters=64, 
                              kernel_size=(5, 5), 
                              activation='relu',padding='same')
        
        self.pool = MaxPooling2D(pool_size=(2, 2), strides=2)
        
        self.flat = Flatten()
        self.dense1 = layers.Dense(100, activation='tanh')
        self.dense2 = layers.Dense(10)
    @tf.function
    def call(self, x):
        h1 = self.l1_conv(x) 
        h1_pool = self.pool(h1) 
        h2 = self.l2_conv(h1_pool)
        h2_pool = self.pool(h2) 
        flat_h = self.flat(h2_pool)
        dense1 = self.dense1(flat_h)
        logits = self.dense2(dense1)
        probs = tf.nn.softmax(logits, axis=-1)
        return probs
    
    @tf.function
    def getL1_feature_map(self, x):
        h1 = self.l1_conv(x) # [32, 28, 28, 32]
        
        return h1
    
    @tf.function
    def getL2_feature_map(self, x):
        h1 = self.l1_conv(x) # [32, 28, 28, 32]
        h1_pool = self.pool(h1) # [32, 14, 14, 32]
        h2 = self.l2_conv(h1_pool) #[32, 14, 14, 64]
        return h2

model = myConvModel()
optimizer = optimizers.Adam(0.001)

## 编译， fit以及evaluate

In [ ]:
import matplotlib.pyplot as plt

model.compile(optimizer=optimizer,
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])
train_ds, test_ds = cifar10_dataset()

# 使用 history 捕获训练和验证过程中的 loss 与 accuracy
history = model.fit(train_ds, validation_data=test_ds, epochs=10)
model.evaluate(test_ds)

# -----------------
# 添加的：绘制CIFAR10的Loss和Accuracy曲线
# -----------------
plt.figure(figsize=(12, 5))

# 绘制 Loss 曲线
plt.subplot(1, 2, 1)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.title('CIFAR-10: Loss over Epochs')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()

# 绘制 Accuracy 曲线
plt.subplot(1, 2, 2)
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Val Accuracy')
plt.title('CIFAR-10: Accuracy over Epochs')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()

plt.tight_layout()
plt.savefig('cifar10_keras_curve.png') # 保存图表至同级目录用于提交报告
plt.show()

In [ ]:
import numpy
import pylab
from PIL import Image
import numpy as np

ds, test_ds = cifar10_dataset()

for i in test_ds:
    test_batch = i[0][:1, :, :]
    break
# 注意：压缩到 32x32 是它变成马赛克的原因，配合CIFAR10所必须
img = Image.open(open('corgi.jpg', 'rb'))
img = numpy.asarray(img, dtype='float32') / 256.
img = np.expand_dims(img, axis=0)

# 获取已训练模型的 L2 卷积层输出矩阵
img_out = model.getL2_feature_map(img)

# ---- 整合为一张大图的形式展示 ----
pylab.figure(figsize=(15, 12))
pylab.suptitle('Trained Model: Original Image and L2 Feature Maps', fontsize=16)

# 1. 绘制原图 (a)
pylab.subplot(5, 5, 1)
pylab.axis('off')
pylab.imshow(img[0, :, :, :])
pylab.title('(a) Original (32x32)', fontsize=10)

# 2. 循环绘制前 20 个特征图 (b) ~ (u)
for i in range(20):
    pylab.subplot(5, 5, i + 2)
    pylab.axis('off')
    pylab.imshow(img_out[0, :, :, i])
    # 利用 ascii 码生成字母标号，98 分别对应 'b', 'c', ...
    pylab.title(f'({chr(98 + i)}) map {i}', fontsize=10)

pylab.tight_layout()
pylab.subplots_adjust(top=0.92) # 为主标题留出空间
pylab.show()

In [ ]:
import numpy
import pylab
from PIL import Image
import numpy as np

# 创建一个没有训练过的新随机模型做对比
rand_model = myConvModel()
ds, test_ds = cifar10_dataset()

for i in test_ds:
    test_batch = i[0][:1, :, :]
    break
# 同样是因为此处重定义到了32*32而导致分辨率下降（马赛克感）
img = Image.open(open('corgi.jpg', 'rb'))
img = numpy.asarray(img, dtype='float32') / 256.
img = np.expand_dims(img, axis=0)

# 获取随机初始化模型的 第一层 卷积层输出矩阵
img_out = rand_model.getL1_feature_map(img)

# ---- 整合为一张大图的形式展示 ----
pylab.figure(figsize=(15, 12))
pylab.suptitle('Random Model: Original Image and L1 Feature Maps', fontsize=16)

# 1. 绘制原图 (a)
pylab.subplot(5, 5, 1)
pylab.axis('off')
pylab.imshow(img[0, :, :, :])
pylab.title('(a) Original (32x32)', fontsize=10)

# 2. 循环绘制前 20 个特征图 (b) ~ (u)
for i in range(20):
    pylab.subplot(5, 5, i + 2)
    pylab.axis('off')
    pylab.imshow(img_out[0, :, :, i])
    pylab.title(f'({chr(98 + i)}) map {i}', fontsize=10)

pylab.tight_layout()
pylab.subplots_adjust(top=0.92) # 为主标题留出空间
pylab.show()